# TP Python 2 : Sélections, Jointures et Requêtes spatiales

**Géoinformatique II — Université de Lausanne**

---

Dans ce TP, tu vas apprendre à effectuer en Python les mêmes opérations que dans QGIS au TP2 :

| Opération QGIS (TP2) | Équivalent Python/GeoPandas |
|---|---|
| Sélection par valeur | Filtrage booléen `gdf[gdf['col'] == val]` |
| Requête par expression SQL | `.query("col == val and col2 > seuil")` |
| Sélection par localisation | `gpd.sjoin()` avec prédicat spatial |
| Jointure attributaire | `.merge()` |
| Jointure spatiale | `gpd.sjoin()` |

Nous travaillerons à nouveau avec **`tp1.gpkg`**, le même fichier GeoPackage qu'au TP1.

In [ ]:
# Importation des bibliothèques
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print(f"GeoPandas : {gpd.__version__}")
print(f"Pandas    : {pd.__version__}")

---
## 1. Chargement et exploration des couches

Commençons par charger les trois couches principales de `tp1.gpkg`.
C'est le même fichier que tu as ouvert dans QGIS au TP1 — mais maintenant en Python !

In [ ]:
# Chargement des trois couches depuis tp1.gpkg
cantons = gpd.read_file('tp1.gpkg', layer='Cantons')
towns   = gpd.read_file('tp1.gpkg', layer='Towns')
lakes   = gpd.read_file('tp1.gpkg', layer='Lakes')

# Résumé pour chaque couche
for nom, gdf in [("Cantons", cantons), ("Towns", towns), ("Lakes", lakes)]:
    print(f"--- {nom} ---")
    print(f"  Entités   : {len(gdf)}")
    print(f"  Colonnes  : {gdf.columns.tolist()}")
    print(f"  Géométrie : {gdf.geom_type.unique()}")
    print(f"  CRS (EPSG): {gdf.crs.to_epsg()}")
    print()

In [ ]:
# Aperçu de la table attributaire (≡ "Ouvrir la table attributaire" dans QGIS)
print("=== Couche Towns ===")
print(towns.head())
print()
print("=== Couche Cantons ===")
print(cantons.head())

---
## 2. Requêtes attributaires

### 2.1 Filtrage booléen

Le filtrage booléen est l'équivalent de l'outil **"Sélectionner les entités par valeur"** dans QGIS.
On construit un masque logique (série de `True`/`False`) qui sélectionne les lignes voulues.

```
gdf[ condition ]
```

Les opérateurs logiques sont :

| Python | Signification |
|--------|---------------|
| `==` | Égal à |
| `!=` | Différent de |
| `>`, `>=` | Supérieur (ou égal) |
| `<`, `<=` | Inférieur (ou égal) |
| `&` | ET logique (entre parenthèses !) |
| `\|` | OU logique (entre parenthèses !) |
| `~` | NON logique (négation) |

In [ ]:
# --- Sélection simple (une seule condition) ---

# Villes avec une population supérieure à 50 000 habitants
grandes_villes = towns[towns['Population'] > 50_000]
print(f"Villes > 50 000 hab. : {len(grandes_villes)}")
print(grandes_villes[['ID1', 'Population']].sort_values('Population', ascending=False).to_string(index=False))
print()

# Villes avec un rang <= 3 (les 3 plus grandes villes)
top3 = towns[towns['Rank'] <= 3]
print(f"Top 3 villes (Rank <= 3) :")
print(top3[['ID1', 'Population', 'Rank']].sort_values('Rank').to_string(index=False))

In [ ]:
# --- Sélections multiples avec & (ET) et | (OU) ---
# Équivalent des requêtes SQL avec AND/OR dans QGIS

# Villes de taille moyenne : entre 20 000 et 50 000 habitants
villes_moyennes = towns[(towns['Population'] >= 20_000) & (towns['Population'] <= 50_000)]
print(f"Villes entre 20 000 et 50 000 hab. : {len(villes_moyennes)}")
print(villes_moyennes[['ID1', 'Population']].sort_values('Population', ascending=False).to_string(index=False))
print()

# Sélection par liste de valeurs avec .isin()
# Équivalent de : "ID1" IN ('Berne', 'Lausanne', 'Genève')
capitales = towns[towns['ID1'].isin(['Bern', 'Lausanne', 'Genf'])]
print(f"Capitales sélectionnées : {len(capitales)}")
print(capitales[['ID1', 'Population']].to_string(index=False))

In [ ]:
# --- Visualisation de la sélection ---
# Dans QGIS, les entités sélectionnées apparaissent en jaune.
# En Python, on utilise deux appels .plot() successifs avec des couleurs différentes.

fig, ax = plt.subplots(figsize=(10, 8))

# Fond : cantons et toutes les villes (gris discret)
cantons.plot(ax=ax, color='#f5f5dc', edgecolor='#888', linewidth=0.7)
towns.plot(ax=ax, color='#aaaaaa', markersize=5, label='Toutes les villes', zorder=2)

# Sélection : grandes villes mises en évidence (rouge)
grandes_villes.plot(
    ax=ax, color='#e53935', markersize=60, marker='*',
    zorder=5, label=f'Population > 50 000 hab. (n={len(grandes_villes)})'
)

# Étiquettes pour les grandes villes
for _, row in grandes_villes.iterrows():
    ax.annotate(
        row['ID1'],
        xy=(row.geometry.x, row.geometry.y),
        xytext=(6, 4), textcoords='offset points',
        fontsize=8, fontweight='bold'
    )

ax.set_title('Sélection par attribut — villes > 50 000 habitants', fontsize=13)
ax.legend(loc='lower right')
ax.set_xlabel('Est (m)')
ax.set_ylabel('Nord (m)')
plt.tight_layout()
plt.show()

### 2.2 La méthode `.query()`

`.query()` accepte une expression sous forme de **chaîne de caractères**, plus lisible
pour des conditions complexes. On peut même y utiliser des variables Python avec le préfixe `@`.

```python
gdf.query("colonne == 'valeur' and colonne2 > seuil")
gdf.query("colonne2 > @variable_python")
```

In [ ]:
# --- Équivalences filtrage booléen ↔ .query() ---

seuil_bas  = 20_000
seuil_haut = 100_000

# Syntaxe booléenne classique
selection_bool = towns[
    (towns['Population'] >= seuil_bas) & (towns['Population'] < seuil_haut)
]

# Même résultat avec .query()  — variable externe précédée de @
selection_query = towns.query("Population >= @seuil_bas and Population < @seuil_haut")

print(f"Filtrage booléen : {len(selection_bool)} villes")
print(f"Via .query()     : {len(selection_query)} villes")
print()

# .query() avec opérateur IN — équivalent de .isin()
noms_cibles = ['Bern', 'Lausanne', 'Genf', 'Basel', 'Zurich']
sel_isin  = towns[towns['ID1'].isin(noms_cibles)]
sel_query = towns.query("ID1 in @noms_cibles")

print("Villes sélectionnées via .query() :")
print(sel_query[['ID1', 'Population', 'Rank']].sort_values('Rank').to_string(index=False))

---
## 3. Calculs géométriques

GeoPandas permet d'accéder directement aux propriétés géométriques de chaque entité :
superficie, périmètre, centroïde, etc. Ces calculs sont **sensibles au CRS** :
en MN03/MN95 (mètres), les résultats sont directement interprétables.

> ⚠️ En WGS84 (degrés), `.area` et `.length` renvoient des valeurs en degrés² ou
> en degrés — peu utiles géographiquement. Toujours vérifier le CRS avant de calculer !

In [ ]:
# --- Surface et périmètre des cantons ---
# CRS = EPSG:21781 (MN03) → coordonnées en mètres → surface en m², périmètre en m

cantons_calc = cantons.copy()
cantons_calc['superficie_km2'] = cantons_calc.geometry.area / 1e6          # m² → km²
cantons_calc['perimetre_km']   = cantons_calc.geometry.length / 1e3        # m  → km

# Affichage des 10 plus grands cantons
top10 = cantons_calc.nlargest(10, 'superficie_km2')
print("10 plus grands cantons :")
print(top10[['NAME', 'superficie_km2', 'perimetre_km']]
      .rename(columns={'NAME': 'Canton', 'superficie_km2': 'Superficie (km²)', 'perimetre_km': 'Périmètre (km)'})
      .to_string(index=False, float_format=lambda x: f"{x:.1f}")
)

In [ ]:
# --- Centroïdes ---
# Le centroïde est le "centre de masse" d'un polygone.
# Utile pour placer des étiquettes ou calculer des distances.

cantons_centroides = cantons.copy()
cantons_centroides['geometry'] = cantons_centroides.geometry.centroid

fig, ax = plt.subplots(figsize=(10, 8))
cantons.plot(ax=ax, color='#e8f4e8', edgecolor='#555', linewidth=0.8)
cantons_centroides.plot(ax=ax, color='#1565c0', markersize=20, zorder=5)

# Étiquette des cantons sur les centroïdes
for _, row in cantons_centroides.iterrows():
    ax.annotate(
        row['NAME'], xy=(row.geometry.x, row.geometry.y),
        ha='center', va='bottom', fontsize=6, color='#1a237e'
    )

ax.set_title('Centroïdes des cantons suisses', fontsize=13)
ax.set_xlabel('Est (m)')
ax.set_ylabel('Nord (m)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Distance entre deux points ---
# Calcul de la distance à vol d'oiseau entre deux villes (en km)

# On travaille en MN03 (mètres) — la distance .distance() est donc en mètres
lausanne = towns[towns['ID1'] == 'Lausanne'].geometry.values[0]
zurich   = towns[towns['ID1'] == 'Zurich'].geometry.values[0]
bern     = towns[towns['ID1'] == 'Bern'].geometry.values[0]

print("Distances à vol d'oiseau (MN03, EPSG:21781) :")
print(f"  Lausanne → Zurich  : {lausanne.distance(zurich) / 1000:.1f} km")
print(f"  Lausanne → Berne   : {lausanne.distance(bern)   / 1000:.1f} km")
print(f"  Berne    → Zurich  : {bern.distance(zurich)     / 1000:.1f} km")
print()

# Distance de chaque ville par rapport à Berne (capitale fédérale)
towns_calc = towns.copy()
towns_calc['dist_berne_km'] = towns_calc.geometry.distance(bern) / 1000
print("Villes les plus éloignées de Berne :")
print(
    towns_calc.nlargest(5, 'dist_berne_km')[['ID1', 'dist_berne_km']]
    .rename(columns={'ID1': 'Ville', 'dist_berne_km': 'Distance (km)'})
    .to_string(index=False, float_format=lambda x: f"{x:.1f}")
)

---
## 4. Requêtes spatiales (prédicats géométriques)

Une **requête spatiale** sélectionne des entités en fonction de leur
**relation géométrique** avec d'autres entités. C'est l'équivalent de
l'outil **"Sélection par localisation"** dans QGIS.

### Prédicats disponibles dans GeoPandas / Shapely

| Prédicat | Description | Exemple |
|---|---|---|
| `intersects` | Ont au moins un point en commun | Route qui traverse un canton |
| `within` | Entièrement contenu dans | Ville dans un canton |
| `contains` | Contient entièrement l'autre | Canton qui contient une ville |
| `overlaps` | Chevauchement partiel (même dimension) | Deux polygones qui se recoupent |
| `crosses` | Traversée (dimensions différentes) | Route qui traverse un lac |
| `touches` | Partagent une frontière mais pas d'intérieur | Cantons voisins |

> 🔗 **Parallèle QGIS** : dans l'outil *Sélection par localisation*, le menu
> déroulant "Where the features" correspond directement aux prédicats ci-dessus.

In [ ]:
# --- Prédicat sur une seule entité ---
# Question : quelles villes se trouvent dans le canton de Vaud ?

# 1. On extrait le polygone du canton de Vaud
vaud = cantons[cantons['NAME'] == 'Vaud'].geometry.values[0]

# 2. On teste pour chaque ville si elle est à l'intérieur (within) du polygone de Vaud
masque_vaud = towns.geometry.within(vaud)

# 3. On applique le masque
villes_vaud = towns[masque_vaud]

print(f"Villes dans le canton de Vaud : {len(villes_vaud)}")
print(villes_vaud[['ID1', 'Population']].sort_values('Population', ascending=False).to_string(index=False))

In [ ]:
# --- Visualisation de la requête spatiale ---
fig, ax = plt.subplots(figsize=(10, 8))

# Tous les cantons (gris clair)
cantons.plot(ax=ax, color='#f0f0f0', edgecolor='#888', linewidth=0.7)

# Canton de Vaud en couleur
cantons[cantons['NAME'] == 'Vaud'].plot(
    ax=ax, color='#a5d6a7', edgecolor='#2e7d32', linewidth=1.5,
    label='Canton de Vaud', zorder=2
)

# Toutes les villes (gris)
towns.plot(ax=ax, color='#bbb', markersize=5, zorder=3, label='Autres villes')

# Villes dans Vaud (vert foncé)
villes_vaud.plot(
    ax=ax, color='#1b5e20', markersize=40, marker='o',
    zorder=5, label=f'Villes dans Vaud (n={len(villes_vaud)})'
)

for _, row in villes_vaud.iterrows():
    ax.annotate(row['ID1'], xy=(row.geometry.x, row.geometry.y),
                xytext=(5, 4), textcoords='offset points', fontsize=7)

ax.set_title('Requête spatiale — villes situées dans le canton de Vaud\n(prédicat : within)', fontsize=12)
ax.legend(loc='lower right')
ax.set_xlabel('Est (m)')
ax.set_ylabel('Nord (m)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Combiner requête attributaire ET requête spatiale ---
# Question : quelles GRANDES villes (> 20 000 hab.) se trouvent dans Vaud ?
# (Équivalent QGIS : d'abord sélection par valeur, puis "ajouter à la sélection" par localisation)

# Étape 1 : requête attributaire
grandes = towns[towns['Population'] > 20_000]
print(f"Étape 1 — villes > 20 000 hab.  : {len(grandes)}")

# Étape 2 : requête spatiale sur la sous-sélection
grandes_vaud = grandes[grandes.geometry.within(vaud)]
print(f"Étape 2 — parmi elles, dans Vaud : {len(grandes_vaud)}")
print(grandes_vaud[['ID1', 'Population']].sort_values('Population', ascending=False).to_string(index=False))

---
## 5. Jointure attributaire — `merge()`

Une **jointure attributaire** combine deux tables en se basant sur une **clé commune**.
C'est l'équivalent de l'outil **"Jointures"** dans les propriétés de couche de QGIS.

```
GeoDataFrame A  ──── clé commune ────  DataFrame B
                        merge()
                           ↓
               GeoDataFrame enrichi (A + colonnes de B)
```

| Paramètre `how=` | Description | Équivalent SQL |
|---|---|---|
| `'left'` | Garde toutes les lignes de gauche | LEFT JOIN |
| `'inner'` | Garde uniquement les correspondances | INNER JOIN |
| `'outer'` | Garde tout, NaN si pas de correspondance | FULL OUTER JOIN |

In [ ]:
# --- Création d'une table de données attributaires supplémentaires ---
# (Équivalent : un fichier CSV/Excel qu'on veut joindre à une couche géographique)
# Données: région linguistique et date d'entrée dans la Confédération par canton

regions_linguistiques = pd.DataFrame({
    'nom_canton': [
        'Zürich', 'Bern', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
        'Nidwalden', 'Glarus', 'Zug', 'Fribourg', 'Solothurn',
        'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen', 'Appenzell Ausserrhoden',
        'Appenzell Innerrhoden', 'St. Gallen', 'Graubünden', 'Aargau',
        'Thurgau', 'Ticino', 'Vaud', 'Valais', 'Neuchâtel', 'Genf', 'Jura'
    ],
    'langue_principale': [
        'Allemand', 'Allemand/Français', 'Allemand', 'Allemand', 'Allemand', 'Allemand',
        'Allemand', 'Allemand', 'Allemand', 'Français/Allemand', 'Allemand',
        'Allemand', 'Allemand', 'Allemand', 'Allemand',
        'Allemand', 'Allemand', 'Romanche/Allemand/Italien', 'Allemand',
        'Allemand', 'Italien', 'Français', 'Français/Allemand', 'Français', 'Français', 'Français'
    ],
    'annee_entree': [
        1351, 1353, 1332, 1291, 1291, 1291,
        1291, 1352, 1352, 1481, 1481,
        1501, 1501, 1501, 1513,
        1513, 1803, 1803, 1803,
        1803, 1803, 1803, 1815, 1815, 1815, 1979
    ]
})

print("Table attributaire créée :")
print(f"  {len(regions_linguistiques)} lignes × {len(regions_linguistiques.columns)} colonnes")
print(regions_linguistiques.head(6))

In [ ]:
# --- Jointure attributaire avec .merge() ---
# On joint la couche 'cantons' (GeoDataFrame) avec 'regions_linguistiques' (DataFrame)
# Clé de jointure : 'NAME' dans cantons  ↔  'nom_canton' dans regions_linguistiques

cantons_enrichis = cantons.merge(
    regions_linguistiques,
    left_on='NAME',           # colonne de clé dans le GeoDataFrame
    right_on='nom_canton',    # colonne de clé dans le DataFrame
    how='left'                # garde tous les cantons, même sans correspondance
)

print(f"Résultat de la jointure : {cantons_enrichis.shape[0]} lignes × {cantons_enrichis.shape[1]} colonnes")
print()
print("Colonnes disponibles après jointure :")
print(cantons_enrichis.columns.tolist())
print()
print("Aperçu :")
print(cantons_enrichis[['NAME', 'langue_principale', 'annee_entree']].dropna().head(8).to_string(index=False))
print()

# Valeurs manquantes ? (cantons sans correspondance dans la table)
manquants = cantons_enrichis[cantons_enrichis['langue_principale'].isna()]
if len(manquants) > 0:
    print(f"⚠ {len(manquants)} canton(s) sans correspondance : {manquants['NAME'].tolist()}")
else:
    print("✓ Tous les cantons ont une correspondance.")

In [ ]:
# --- Visualisation : carte choroplèthe par région linguistique ---
# Après la jointure, on peut visualiser le nouvel attribut sur la carte

# Palette de couleurs par langue
palette = {
    'Allemand':               '#4fc3f7',
    'Français':               '#ffb74d',
    'Italien':                '#81c784',
    'Français/Allemand':      '#ffcc02',
    'Allemand/Français':      '#ffcc02',
    'Romanche/Allemand/Italien': '#ce93d8',
}

fig, ax = plt.subplots(figsize=(10, 8))

for langue, couleur in palette.items():
    sous_gdf = cantons_enrichis[cantons_enrichis['langue_principale'] == langue]
    if len(sous_gdf) > 0:
        sous_gdf.plot(ax=ax, color=couleur, edgecolor='#555', linewidth=0.8,
                      label=langue, zorder=2)

# Cantons sans info (NaN) en gris
cantons_enrichis[cantons_enrichis['langue_principale'].isna()].plot(
    ax=ax, color='#eee', edgecolor='#888', linewidth=0.8, label='Non défini', zorder=2
)

ax.set_title('Cantons suisses par région linguistique principale\n(après jointure attributaire)', fontsize=12)
ax.legend(loc='lower right', fontsize=8, title='Langue', title_fontsize=9)
ax.set_xlabel('Est (m)')
ax.set_ylabel('Nord (m)')
plt.tight_layout()
plt.show()

---
## 6. Jointure spatiale — `gpd.sjoin()`

La **jointure spatiale** associe deux couches sur la base de leur relation géométrique
plutôt que d'une clé commune. C'est l'équivalent de l'outil
**"Joindre les attributs par localisation"** dans QGIS.

```python
gpd.sjoin(
    gdf_gauche,          # GeoDataFrame qui reçoit les attributs
    gdf_droit,           # GeoDataFrame qui fournit les attributs
    how='left',          # type de jointure (left / inner / right)
    predicate='within'   # relation spatiale
)
```

> 🔗 **Exemple** : associer à chaque ville le nom du canton dans lequel elle se trouve.

In [ ]:
# --- Jointure spatiale : assigner le canton à chaque ville ---
# Objectif : pour chaque ville (point), retrouver le canton (polygone) qui la contient

# sjoin — predicate='within' : chaque point est associé au polygone qui le contient
towns_avec_canton = gpd.sjoin(
    towns[['ID1', 'Population', 'Rank', 'geometry']],   # couche de points
    cantons[['NAME', 'geometry']],                       # couche de polygones
    how='left',             # garde toutes les villes (même hors des cantons)
    predicate='within'      # prédicat : point contenu dans polygone
)

# Renommage pour plus de clarté
towns_avec_canton = towns_avec_canton.rename(columns={'NAME': 'canton'})

print(f"Villes avec leur canton attribué : {len(towns_avec_canton)}")
print()
print("Aperçu :")
print(towns_avec_canton[['ID1', 'Population', 'canton']]
      .sort_values('Population', ascending=False).head(10).to_string(index=False))
print()

# Villes sans canton attribué (peuvent se trouver sur une frontière)
sans_canton = towns_avec_canton[towns_avec_canton['canton'].isna()]
if len(sans_canton) > 0:
    print(f"⚠ {len(sans_canton)} ville(s) sans canton : {sans_canton['ID1'].tolist()}")

### 6.1 Agrégation après jointure spatiale

Une fois la jointure spatiale effectuée, on peut **agréger** les données pour
calculer des statistiques par canton : nombre de villes, population totale, etc.
C'est l'équivalent de la combinaison **jointure + calculatrice de champs** dans QGIS.

In [ ]:
# --- Agrégation par canton après jointure spatiale ---

# Nombre de villes et population urbaine totale par canton
stats_cantons = (
    towns_avec_canton
    .dropna(subset=['canton'])          # exclure les villes sans canton
    .groupby('canton')
    .agg(
        nb_villes   = ('ID1',        'count'),
        pop_urbaine = ('Population', 'sum'),
        pop_max     = ('Population', 'max'),
        rang_min    = ('Rank',       'min'),    # rang le plus élevé = ville la plus grande
    )
    .reset_index()
    .sort_values('pop_urbaine', ascending=False)
)

print("Statistiques urbaines par canton (Top 10 par population urbaine) :")
print(stats_cantons.head(10).to_string(index=False))

In [ ]:
# --- Jointure de stats_cantons sur le GeoDataFrame cantons ---
# On combine la table d'agrégation avec la géométrie des cantons pour faire une carte

cantons_stats = cantons.merge(
    stats_cantons,
    left_on='NAME', right_on='canton',
    how='left'
)

# Carte du nombre de villes par canton
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Carte 1 : nombre de villes
cantons_stats.plot(
    ax=axes[0], column='nb_villes',
    cmap='YlOrBr', scheme='natural_breaks', k=4,
    legend=True, edgecolor='white', linewidth=0.5,
    missing_kwds={'color': '#ddd', 'label': 'Aucune ville recensée'}
)
axes[0].set_title('Nombre de villes par canton', fontsize=12)
axes[0].set_axis_off()

# Carte 2 : population urbaine totale
cantons_stats.plot(
    ax=axes[1], column='pop_urbaine',
    cmap='Blues', scheme='quantiles', k=4,
    legend=True, edgecolor='white', linewidth=0.5,
    missing_kwds={'color': '#ddd', 'label': 'Aucune ville recensée'}
)
axes[1].set_title('Population urbaine totale par canton', fontsize=12)
axes[1].set_axis_off()

plt.suptitle('Résultat de la jointure spatiale + agrégation', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Relations 1-N : dissolve et agrégation

Dans QGIS, les **relations** permettent de naviguer entre des entités liées
(ex. : une parcelle et ses propriétaires). En Python, on utilise `groupby()` + `dissolve()`
pour des agrégations et fusions de géométries.

`dissolve()` **fusionne** les polygones qui partagent la même valeur d'un attribut,
un peu comme un `groupby` spatial.

> 🔗 **Exemple** : regrouper les cantons par région linguistique pour créer de nouvelles entités.

In [ ]:
# --- Dissolve : fusionner les cantons par région linguistique ---
# 1. On simplifie la colonne langue pour n'avoir que 3 grandes régions
def region_simple(langue):
    if langue is None or pd.isna(langue):
        return 'Inconnue'
    if 'Italien' in str(langue) and 'Romanche' not in str(langue):
        return 'Italophone'
    if 'Romanche' in str(langue):
        return 'Plurilingue'
    if 'Français' in str(langue) and 'Allemand' not in str(langue):
        return 'Francophone'
    return 'Germanophone'          # Allemand pur ou Allemand/Français

cantons_enrichis['region']        = cantons_enrichis['langue_principale'].apply(region_simple)
cantons_enrichis['superficie_km2'] = cantons_enrichis.geometry.area / 1e6

# 2. Dissolve : fusionne les géométries par région
regions = cantons_enrichis.dissolve(
    by='region',
    aggfunc={
        'NAME':          'count',   # nombre de cantons par région
        'superficie_km2': 'sum',    # superficie totale (m² → km² recalculé après)
    }
).reset_index()

# Recalcul précis de la superficie depuis la géométrie fusionnée
regions['superficie_km2'] = regions.geometry.area / 1e6

print("Régions linguistiques après dissolve :")
print(regions[['region', 'NAME', 'superficie_km2']]
      .rename(columns={'NAME': 'nb_cantons', 'superficie_km2': 'superficie_km2'})
      .to_string(index=False, float_format=lambda x: f"{x:.0f}"))

In [ ]:
# --- Visualisation des régions issues du dissolve ---
palette_regions = {
    'Germanophone': '#4fc3f7',
    'Francophone':  '#ffb74d',
    'Italophone':   '#81c784',
    'Plurilingue':  '#ce93d8',
    'Inconnue':     '#cccccc',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Carte gauche : cantons colorés par région linguistique (avant dissolve)
for region, couleur in palette_regions.items():
    sous = cantons_enrichis[cantons_enrichis['region'] == region]
    if len(sous) > 0:
        sous.plot(ax=axes[0], color=couleur, edgecolor='white', linewidth=0.6, label=region, zorder=2)
cantons.plot(ax=axes[0], color='none', edgecolor='#555', linewidth=0.3, zorder=3)
axes[0].set_title('Cantons par région linguistique', fontsize=12)
axes[0].legend(loc='lower right', fontsize=8)
axes[0].set_axis_off()

# Carte droite : polygones fusionnés après dissolve
for region, couleur in palette_regions.items():
    sous = regions[regions['region'] == region]
    if len(sous) > 0:
        sous.plot(ax=axes[1], color=couleur, edgecolor='#333', linewidth=1.2, label=region, zorder=2)
axes[1].set_title('Régions linguistiques (après dissolve)', fontsize=12)
axes[1].legend(loc='lower right', fontsize=8)
axes[1].set_axis_off()

plt.suptitle('dissolve() — fusion de polygones par attribut commun', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Récapitulatif — QGIS ↔ Python

| Opération QGIS (TP2) | Code Python / GeoPandas |
|---|---|
| Sélectionner par valeur | `gdf[gdf['col'] == 'val']` |
| Requête SQL / expression | `gdf.query("col == 'val' and col2 > seuil")` |
| Opérateur IN | `gdf[gdf['col'].isin(['a', 'b', 'c'])]` |
| Sélection par localisation | `gdf[gdf.geometry.within(polygon)]` |
| Calculatrice de champs — surface | `gdf.geometry.area / 1e6` (→ km²) |
| Calculatrice de champs — périmètre | `gdf.geometry.length / 1e3` (→ km) |
| Distance entre entités | `geom_a.distance(geom_b) / 1000` (→ km) |
| Centroïde | `gdf.geometry.centroid` |
| Jointure attributaire | `gdf.merge(df, left_on='cle', right_on='cle', how='left')` |
| Jointure spatiale (par localisation) | `gpd.sjoin(gdf_pts, gdf_poly, how='left', predicate='within')` |
| Statistiques par groupe (après jointure) | `gdf.groupby('canton').agg({'pop': 'sum'})` |
| Fusionner des polygones (dissolve) | `gdf.dissolve(by='attribut')` |
| Sauvegarder une couche | `gdf.to_file("sortie.gpkg", layer="nom", driver="GPKG")` |

In [ ]:
import os

# Création du dossier de sortie
os.makedirs("sortie_tp02", exist_ok=True)

# Exportation des résultats dans un GeoPackage
# (plusieurs couches dans un seul fichier — recommandé)

towns_avec_canton.to_file(
    "sortie_tp02/resultats_tp02.gpkg",
    layer="villes_avec_canton", driver="GPKG"
)

cantons_enrichis.to_file(
    "sortie_tp02/resultats_tp02.gpkg",
    layer="cantons_avec_langue", driver="GPKG"
)

regions.to_file(
    "sortie_tp02/resultats_tp02.gpkg",
    layer="regions_linguistiques", driver="GPKG"
)

print("Fichiers exportés dans sortie_tp02/resultats_tp02.gpkg :")
print("  - villes_avec_canton    (jointure spatiale)")
print("  - cantons_avec_langue   (jointure attributaire)")
print("  - regions_linguistiques (dissolve)")
print()
print("Tu peux ouvrir ce fichier directement dans QGIS pour vérifier les résultats !")